# Project 2: Traffic Vehicle Classification — Completed

## Reliable traffic-camera classification with CNNs, transfer learning, and human review

This notebook is the executable companion to the original project definition. It audits the data before splitting, prevents cross-split leakage, performs the required controlled experiments, selects a production model on validation data, evaluates the frozen test set once, analyzes unseen/uncertain cases, and provides review-aware JSON inference.

> **Outcome:** a complete reproducible project, not a demonstration with an unevaluated training loop. All measured values below come from saved artifacts in `artifacts/` and `reports/`.

Start with `README.md` for setup and `reports/final_report.md` for the complete written analysis.

## 1. Reproducible setup

The code imports the tested implementation from `traffic_classifier/`. If all result files are absent, the workflow cell trains every experiment. Otherwise it reuses completed checkpoints/results and presents the saved evidence.

In [1]:
from pathlib import Path
import json
import platform
import numpy as np
import pandas as pd
import torch
import torchvision
import sklearn
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / "traffic_classifier").exists():
    ROOT = ROOT.parent

required = [
    ROOT / "artifacts/results/experiment_results.json",
    ROOT / "artifacts/results/frozen_test_evaluation.json",
    ROOT / "artifacts/checkpoints/final_model.pt",
]
print({
    "python": platform.python_version(),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "sklearn": sklearn.__version__,
    "cuda_available": torch.cuda.is_available(),
    "project_root": str(ROOT),
})

{'python': '3.14.6', 'torch': '2.14.1+cu130', 'torchvision': '0.29.1+cu130', 'sklearn': '1.9.1', 'cuda_available': True, 'project_root': 'C:\\Users\\viroo\\PycharmProjects\\TrafficVehicle'}


## 2. Audit before validation and before model fitting

The audit verifies counts, formats, readability, dimensions, quality issues, encoded-file hashes, and **decoded RGB pixel hashes**. Both hash methods identify the same duplicate groups in this supplied copy; the canonical pixel hash is encoding-independent by design. No train or test item is silently removed; duplicate copies inside `unclean` are excluded from the separate data-quality analysis.

In [2]:
from traffic_classifier.pipeline import run_project

if not all(path.exists() for path in required):
    run_project(ROOT / "configs/default.json")

audit_summary = json.loads((ROOT / "artifacts/audit/summary.json").read_text(encoding="utf-8"))
audit_frame = pd.DataFrame([
    {
        "split": split,
        "images": values["images"],
        "width_min": values["width_quantiles"]["min"],
        "width_median": values["width_quantiles"]["median"],
        "width_max": values["width_quantiles"]["max"],
        "height_min": values["height_quantiles"]["min"],
        "height_median": values["height_quantiles"]["median"],
        "height_max": values["height_quantiles"]["max"],
        "unique_pixels": values["unique_pixel_sha256"],
    }
    for split, values in audit_summary["splits"].items()
])
display(audit_frame)
print("Class counts:", {k: v["class_counts"] for k, v in audit_summary["splits"].items()})
print("Quality findings:", audit_summary["issue_counts"] or "No unreadable, unsupported, unusually small, or extreme-aspect files")
print("Canonical duplicate groups:", audit_summary["canonical_duplicate_groups"])
print("Cross-split duplicate groups:", audit_summary["cross_split_duplicate_groups"])
print("Label-conflict groups:", audit_summary["label_conflict_duplicate_groups"])
print("Clean counts:", audit_summary["cleaning"]["clean_counts"])

Device: cuda

=== 1. Dataset audit ===

=== 2. Leakage-safe fixed validation split ===
Run fingerprint: f7dd75aef87f13cbd595b5a743d0a11917fc23db3745b3456cb256a33bc1ee03

=== 3. Controlled experiments (validation only) ===
Reusing completed experiment: cnn_baseline
Reusing completed experiment: no_augmentation
Reusing completed experiment: dropout_0_3
Reusing completed experiment: dropout_0_5
Reusing completed experiment: avg_pool
Reusing completed experiment: weight_decay_1e-4
Reusing completed experiment: step_lr
Reusing completed experiment: bce_loss
Reusing completed experiment: best_regularized
Reusing completed experiment: imbalanced_standard
Reusing completed experiment: imbalanced_balanced
Reusing completed experiment: resnet18_feature_extraction
Reusing completed experiment: resnet18_fine_tuning

=== 4. Select final model from validation only ===
Selected resnet18_fine_tuning with validation macro-F1 0.9138

=== 5. One frozen-test evaluation ===


RuntimeError: A frozen-test result already exists for a different model, protocol, or evidence hash. Refusing to score test again implicitly; use a new artifacts directory or pass --force only for an intentional new evaluation.

In [ ]:
display(Image(filename=str(ROOT / "reports/figures/representative_images.png"), width=950))

The gallery contains 16 reproducible examples—two from each known class—and is generated from the exact paths saved in `artifacts/audit/representative_paths.json`.

In [ ]:
display(Image(filename=str(ROOT / "reports/figures/image_size_distributions.png"), width=950))

## 3. Leakage-safe split, transforms, and reproducibility

The split is stratified by class with seed 42. Exact sample indices and paths are stored before training. Augmentation is present only in the training transform; validation/test use deterministic resize, `ToTensor`, and ImageNet normalization.

In [ ]:
split = json.loads((ROOT / "artifacts/splits/train_validation.json").read_text(encoding="utf-8"))
print("Seed:", split["seed"])
print("Validation fraction:", split["validation_fraction"])
print("Train / validation:", split["train_count"], "/", split["validation_count"])
print("Class mapping:", split["class_to_idx"])
print("Frozen test:", split["frozen_test"])
print("Training transform:", json.dumps(split["transform"], indent=2))
for record in split["by_class"]:
    print(record["class_name"], "train:", len(record["train_indices"]), "validation:", len(record["validation_indices"]))

## 4. CNN baseline and complete training loop

`TrafficCNN` has four convolution–BatchNorm–ReLU blocks, pooling, adaptive average pooling, optional dropout, and an eight-class linear head. Checkpoints are selected by validation macro-F1. Every epoch records training/validation loss, train–validation gap, validation accuracy/precision/recall/F1, stage, and all learning-rate groups.

In [ ]:
from traffic_classifier.models import TrafficCNN
from traffic_classifier.data import build_transforms
from traffic_classifier.config import ProjectConfig

config = ProjectConfig.from_json(ROOT / "configs/default.json")
train_transform, train_transform_spec = build_transforms(config, augmentation=True, kind="cnn")
eval_transform, eval_transform_spec = build_transforms(config, augmentation=False, kind="cnn")
print("Train transform:\n", train_transform)
print("Eval transform:\n", eval_transform)

model = TrafficCNN(num_classes=8, dropout=0.3, pooling="max")
print(model)
print("Parameters:", sum(parameter.numel() for parameter in model.parameters()))

## 5. Custom `BalancedBatchSampler`

The original training data are balanced, so a reproducible simulated imbalance retains 14 examples from `kamyun`, `minibus`, `savari`, and `vanet`, while retaining 40 from each other class. The sampler oversamples to equal class exposure and places exactly four examples of every class in every size-32 batch. Standard and balanced loaders receive the same number of optimizer steps.

In [ ]:
from traffic_classifier.data import (
    load_image_folder, stratified_validation_indices, simulated_imbalance_indices,
    BalancedBatchSampler,
)

base_transform, _ = build_transforms(config, augmentation=True, kind="cnn")
base_train = load_image_folder(ROOT / "dataset", "train", base_transform, config.classes)
train_idx, val_idx, _ = stratified_validation_indices(base_train, 0.20, config.seed)
imbalanced_idx, imbalance_detail = simulated_imbalance_indices(
    train_idx, base_train.targets, base_train.class_to_idx,
    config.imbalanced_classes, config.imbalanced_retained_per_minority_class, config.seed + 100,
)
sampler = BalancedBatchSampler(base_train.targets, imbalanced_idx, 8, 32, seed=config.seed)
sampler.set_epoch(0)
first_batch = next(iter(sampler))
counts = np.bincount(np.asarray(base_train.targets)[first_batch], minlength=8)
print("Retained class counts:", imbalance_detail["class_counts"])
print("Batches / examples per epoch:", len(sampler), sampler.total_size)
print("First batch class counts:", dict(zip(config.classes, counts)))
print("First batch size:", len(first_batch))
print("Exact retained indices are saved in artifacts/splits/simulated_imbalance.json")

## 6. All controlled validation experiments

Single-factor comparisons use the same seed, split, architecture capacity, optimizer, LR, and epoch budget. The explicit regularized combination is identified as such. The frozen test is not involved in any row below.

In [ ]:
experiment_results = json.loads((ROOT / "artifacts/results/experiment_results.json").read_text(encoding="utf-8"))
experiment_summary = pd.read_csv(ROOT / "artifacts/results/experiment_summary.csv")
display(experiment_summary)

### Required main comparison

The CE/BCE row below uses the BCE result only to identify that controlled loss comparison; the dedicated CE baseline appears in the baseline row. A complete side-by-side CE/BCE table follows.

In [ ]:
from traffic_classifier.metrics import worst_classes

main_order = [
    "cnn_baseline", "imbalanced_balanced", "bce_loss", "best_regularized",
    "resnet18_feature_extraction", "resnet18_fine_tuning",
]
main_rows = []
for name in main_order:
    metrics = experiment_results[name]["validation_metrics"]
    low_recall = worst_classes(metrics, "recall")
    low_precision = worst_classes(metrics, "precision")
    main_rows.append({
        "Experiment": experiment_results[name]["display_name"],
        "Macro Precision": metrics["macro_precision"],
        "Macro Recall": metrics["macro_recall"],
        "Macro F1": metrics["macro_f1"],
        "Lowest-Recall Class": f"{low_recall[0]} ({low_recall[1]:.3f})",
        "Lowest-Precision Class": f"{low_precision[0]} ({low_precision[1]:.3f})",
    })
display(pd.DataFrame(main_rows).round(4))

In [ ]:
display(Image(filename=str(ROOT / "reports/figures/training_curves.png"), width=1000))

### Per-class comparison

This heatmap-style table contains validation precision, recall, and F1 for all main experiments.

In [ ]:
display(Image(filename=str(ROOT / "reports/figures/per_class_comparison.png"), width=1000))

## 7. Balanced-batch effect on every class

Macro-F1 and per-class recall are the primary fairness measures. Overall accuracy can fall because a balanced sampler intentionally sacrifices majority prevalence.

In [ ]:
balanced_comparison = json.loads((ROOT / "artifacts/results/balanced_batch_comparison.json").read_text(encoding="utf-8"))
balance_rows = []
for class_name in config.classes:
    standard = balanced_comparison["standard"]["per_class"][class_name]
    balanced = balanced_comparison["balanced"]["per_class"][class_name]
    balance_rows.append({
        "class": class_name,
        "standard_recall": standard["recall"],
        "balanced_recall": balanced["recall"],
        "recall_delta": balanced["recall"] - standard["recall"],
        "standard_precision": standard["precision"],
        "balanced_precision": balanced["precision"],
        "standard_f1": standard["recall"] * 0 + standard["f1"],
        "balanced_f1": balanced["f1"],
    })
display(pd.DataFrame(balance_rows).round(4))
print("Standard macro-F1:", balanced_comparison["standard"]["macro_f1"])
print("Balanced macro-F1:", balanced_comparison["balanced"]["macro_f1"])
print("Equal examples/class/batch:", balanced_comparison["batch_size"] // 8)
print("Same steps/epoch:", balanced_comparison["standard_and_balanced_steps_per_epoch"])

## 8. Cross-entropy vs. BCEWithLogitsLoss

BCE uses one-hot `float32` targets, no pre-loss sigmoid, and `argmax(logits)` for classification. Cross-entropy models the mutually exclusive task directly and is the only eligible production loss. BCE sigmoid scores are independent and do not form a normalized probability distribution.

In [ ]:
loss_rows = []
for name in ("cnn_baseline", "bce_loss"):
    result = experiment_results[name]
    metrics = result["validation_metrics"]
    uncertainty = result["validation_uncertainty"]
    loss_rows.append({
        "experiment": result["display_name"],
        "accuracy": metrics["accuracy"],
        "macro_precision": metrics["macro_precision"],
        "macro_recall": metrics["macro_recall"],
        "macro_f1": metrics["macro_f1"],
        "mean_top_score": uncertainty["mean_confidence"],
        "score_interpretation": uncertainty["probability_interpretation"],
    })
display(pd.DataFrame(loss_rows).round(4))
for name in ("cnn_baseline", "bce_loss"):
    print("\n", experiment_results[name]["display_name"])
    display(pd.DataFrame(experiment_results[name]["validation_metrics"]["per_class"]).T.round(4))

## 9. ResNet18 feature extraction and fine-tuning

Both strategies retain standard `conv1` and `maxpool`, ImageNet normalization, and the same total training budget. Feature extraction freezes the backbone. Fine-tuning first trains the new head, then unfreezes `layer4` with a smaller learning rate. Parameter groups and every epoch's LRs are recorded.

In [ ]:
transfer_rows = []
for name in ("resnet18_feature_extraction", "resnet18_fine_tuning"):
    result = experiment_results[name]
    report = result["metadata"]["parameter_report_head_stage"]
    transfer_rows.append({
        "experiment": result["display_name"],
        "total_parameters": report["total_parameters"],
        "head_stage_trainable": report["trainable_parameters"],
        "head_stage_frozen": report["frozen_parameters"],
        "trainable_percent": report["trainable_percent"],
        "best_epoch": result["best_epoch"],
        "validation_macro_f1": result["validation_metrics"]["macro_f1"],
    })
display(pd.DataFrame(transfer_rows).round(4))

fine_history = pd.DataFrame([
    {
        "epoch": row["epoch"],
        "stage": row["stage"],
        "learning_rates": str(row["learning_rates"]),
        "trainable_parameters": str(row["trainable_parameters"]),
        "validation_macro_f1": row["validation_macro_f1"],
    }
    for row in experiment_results["resnet18_fine_tuning"]["history"]
])
display(fine_history)

## 10. Validation-only model selection and uncertainty

The production candidate with maximum validation macro-F1 is chosen before test. A scalar temperature is fit on validation logits, then the review threshold is the lowest confidence threshold reaching at least 80% automatic coverage. The resulting operational policy sends the remaining low-confidence volume to human review.

In [ ]:
uncertainty = json.loads((ROOT / "artifacts/results/validation_uncertainty.json").read_text(encoding="utf-8"))
final_result = json.loads((ROOT / "artifacts/results/frozen_test_evaluation.json").read_text(encoding="utf-8"))
print("Selected experiment:", final_result["selected_experiment"])
print("Validation-selected temperature:", uncertainty["temperature"])
print("Review threshold:", uncertainty["review_threshold"]["threshold"])
print("Target automatic coverage:", uncertainty["review_threshold"]["target_coverage"])
print("Validation calibration before:", uncertainty["uncalibrated"])
print("Validation calibration after:", uncertainty["calibrated"])
display(Image(filename=str(ROOT / "reports/figures/validation_risk_coverage.png"), width=700))

## 11. One frozen-test evaluation

The test set is now shown only because the selection record above is complete. It reports both confusion forms, all macro/per-class metrics, top mutual confusions, and low-confidence errors. It is not used to revise the model or threshold.

In [ ]:
test_metrics = final_result["test_metrics"]
test_overall = pd.DataFrame([{
    "selected_model": final_result["selected_experiment"],
    "validation_macro_f1": final_result["selection_validation_metrics"]["macro_f1"],
    "test_accuracy": test_metrics["accuracy"],
    "test_macro_precision": test_metrics["macro_precision"],
    "test_macro_recall": test_metrics["macro_recall"],
    "test_macro_f1": test_metrics["macro_f1"],
    "test_ece": final_result["test_uncertainty"]["ece_10_bins"],
    "test_review_rate": final_result["test_selective"]["review_rate"],
}])
display(test_overall.round(4))
display(pd.DataFrame(test_metrics["per_class"]).T.round(4))
display(pd.DataFrame(final_result["mutual_confusion_pairs"]).head(10).round(4))

In [ ]:
display(Image(filename=str(ROOT / "reports/figures/final_confusion_matrices.png"), width=1000))

In [ ]:
display(Image(filename=str(ROOT / "reports/figures/misclassified_test_images.png"), width=1000))

In [ ]:
test_errors = pd.read_csv(ROOT / "artifacts/results/test_errors.csv")
display(test_errors.head(12))

### Confusion and merge decision

The pair score is `C_norm[i,j] + C_norm[j,i]`. A large score triggers inspection, not an automatic merge. The final report assesses actual images, support, and whether the labels have distinct operational meaning before deciding whether to retain the eight-class taxonomy.

In [ ]:
top_pair = final_result["mutual_confusion_pairs"][0]
print("Strongest mutual pair:", top_pair)
print("Decision: retain the eight-class production taxonomy unless image inspection demonstrates redundant operational labels; confusion alone is insufficient.")

## 12. `unclean` and unseen `neysan`

The final model is applied to `unclean` only after removing all pixel-identical train/test copies. `neysan` is never treated as a ninth class. It is an unseen/human-review case. High known-class confidence is not proof against distribution shift.

In [ ]:
unclean_result = json.loads((ROOT / "artifacts/results/unclean_analysis.json").read_text(encoding="utf-8"))
unclean_predictions = pd.read_csv(ROOT / "artifacts/results/unclean_predictions.csv")
print("Cleaning:", unclean_result["cleaning"])
print("Known-class metrics:", unclean_result["known_class_metrics"])
print("Unseen neysan analysis:", unclean_result["unseen_neysan"])
display(unclean_predictions[unclean_predictions["is_unseen_neysan"]].sort_values("confidence").head(12))
display(Image(filename=str(ROOT / "reports/figures/unclean_low_confidence_examples.png"), width=1000))

## 13. Reproducible JSON prediction

The callable loads architecture, class order, transform metadata, temperature, threshold, seed, and weights from the final checkpoint. Probabilities are Cross-Entropy softmax outputs.

In [ ]:
from traffic_classifier.predict import predict_image

sample_image = sorted((ROOT / "dataset/test/ambulance").glob("*.jpg"))[0]
prediction = predict_image(sample_image, checkpoint_path=ROOT / "artifacts/checkpoints/final_model.pt")
print("Image:", sample_image)
print(json.dumps(prediction, indent=2))
assert set(prediction["probabilities"]) == set(config.classes)
assert abs(sum(prediction["probabilities"].values()) - 1.0) < 1e-3
assert isinstance(prediction["needs_review"], bool)

## 14. Conclusions and limitations

- The audit detects metadata-insensitive pixel duplicates across `unclean` and frozen splits, including a `vanet`/`neysan` label conflict.
- Controlled ablations use validation only; the combined regularized model and transfer strategy are interpreted with the small 80-image validation uncertainty in mind.
- Balanced batches improve fairness only when the operational objective values minority recall/macro-F1 rather than prevalence-weighted accuracy alone.
- Cross-entropy is retained for production because the classes are mutually exclusive and valid probabilities should sum to one.
- Human review mitigates uncertainty but does not make a tiny, cropped-image dataset error-free.
- Full class metrics, epoch histories, checkpoints metadata, errors, and transformed data evidence are saved rather than hidden in notebook state.

## 15. Final checklist

- [x] Identical class mapping asserted for train and test.
- [x] Audit completed before validation creation and frozen test use.
- [x] Test used only once for final evaluation.
- [x] Cross-split duplicates detected with decoded image-content hashes.
- [x] Augmentation applied only to training.
- [x] Reproducible simulated imbalance and exact retained indices recorded.
- [x] Balanced batch has equal examples from all eight classes.
- [x] BCE uses one-hot float targets and argmax logits.
- [x] Scheduler uses validation behavior; per-epoch LRs are logged.
- [x] Accuracy, precision, recall, F1, count/normalized confusions reported.
- [x] Per-class metrics and lowest classes compared across experiments.
- [x] At least 12 frozen-test errors plotted with labels/confidence.
- [x] `unclean` analyzed separately after duplicate removal.
- [x] `neysan` retained as unseen/review analysis, not a training class.
- [x] Final checkpoint contains mapping, transform, threshold, temperature, seed, and config.
- [x] JSON prediction and one-command reproduction are provided.

---

# Appendix: Original Project Definition and Grading Rubric

The original assignment cells are retained unchanged below so the completed work remains directly traceable to every requirement.

# Project 2: Traffic Vehicle Classification

## Reliable Traffic-Vehicle Classification with CNNs and Transfer Learning

This project moves from tabular classical machine learning to computer vision. You will build, evaluate, and critically examine an image classifier for cropped traffic-camera vehicles.

> **Scope:** All material through the end of Week 11. 

## Central Question

> Can we build a reliable system that classifies vehicles from traffic-camera images, explains its failures, and recommends human review for uncertain predictions?

The goal is not only accuracy. The final system should be reproducible, robust to data-quality issues, and transparent about uncertainty.

## Dataset

Each image is a crop of one vehicle from a traffic-camera frame. The expected layout is:

| Split | Classes | Intended use |
|---|---|---|
| `dataset/train` | `ambulance`, `autobus`, `kamyun`, `kamyunet`, `minibus`, `savari`, `taxi`, `vanet` | Training and validation source |
| `dataset/test` | The same 8 classes | Final frozen evaluation |
| `dataset/unclean` | The 8 known classes plus `neysan` | Data-quality and unseen-class analysis |

The supplied listing suggests roughly 400 train, 400 test, and 450 unclean images. Verify all counts programmatically; do not hard-code them. The original train split appears class-balanced.

## Data Quality and Evaluation Protocol

Do **not** merge `unclean` directly into training. The supplied listing contains repeated filenames across splits and at least one apparent label conflict. Filename equality is not enough: use an image-content hash to identify real duplicates.

Follow this order:

1. Find unreadable files, unsupported formats, unusually small images, and unusual image dimensions.
2. Report class counts, image-size distributions, and representative examples.
3. Detect duplicate images with hashes across `train`, `test`, and `unclean`; document conflicts.
4. Create cleaned splits and document every exclusion decision.
5. Create a stratified validation split from cleaned training data using a fixed seed; 80/20 is a recommended starting point.
6. Freeze the cleaned test set. Use validation for all model, threshold, scheduler, and hyperparameter choices; use test once for final evaluation.

## Learning Outcomes

By the end of this project, you should be able to:

- Audit an image dataset and load it with `ImageFolder` and `DataLoader`.
- Design transforms for resize, normalization, and augmentation.
- Build a CNN baseline and a complete PyTorch training loop.
- Implement and evaluate a custom `BalancedBatchSampler`.
- Compare `CrossEntropyLoss` and `BCEWithLogitsLoss` for a single-label task.
- Evaluate dropout, weight decay, pooling, augmentation, and learning-rate scheduling.
- Use pretrained ResNet18 for feature extraction and fine-tuning.
- Compare models with per-class precision, recall, F1, and confusion matrices.
- Analyze errors, class relationships, and low-confidence predictions.
- Deliver a reproducible prediction script with JSON output.

## Required Technical Work

### 1. Data audit and baseline
- Report counts for original and cleaned splits, and display at least 12 labelled images.
- Record the random seed, validation indices, and every transform.
- Implement a CNN with at least two convolution + activation + pooling blocks.
- Record training loss, validation loss, and validation metrics at every epoch. Save the best checkpoint by a validation metric.

### 2. Controlled ablations
Change one factor at a time, unless the goal is explicitly to test a combination. Use the same baseline, split, and training budget for each one-factor comparison.

| Experiment | Question |
|---|---|
| No augmentation vs. augmentation | Does realistic variation improve generalization? |
| Dropout: `p=0`, `0.3`, or `0.5` | Does dropout reduce overfitting? |
| Max pooling vs. average pooling | Which spatial summary is more useful? |
| `weight_decay=0` vs. `1e-4` | Does weight regularization improve validation behaviour? |
| Fixed learning rate vs. scheduler | Does scheduling improve convergence or validation performance? |
| Standard batches vs. balanced batches | How does sampling affect recall under imbalance? |
| Cross-entropy vs. BCE | How do the loss assumptions affect a single-label problem? |
| Feature extraction vs. fine-tuning | Does adapting a pretrained backbone help? |

## Loading the Dataset Correctly

Do not use `ImageFolder("dataset")`: it would interpret `train`, `test`, and `unclean` as class names. Load each split separately:

```python
from torchvision import datasets

train_dataset = datasets.ImageFolder("dataset/train", transform=train_transform)
test_dataset = datasets.ImageFolder("dataset/test", transform=eval_transform)
assert train_dataset.class_to_idx == test_dataset.class_to_idx
```

Load `unclean` separately. Because it contains `neysan`, it is not an ordinary 8-class test split.

## Balanced Batches on Simulated Imbalance

The original train split appears balanced, so a balanced sampler would have little visible effect. Create a **reproducible simulated imbalance** from cleaned training data: with a fixed seed, retain fewer examples from selected classes and report the retained indices. Do not use `unclean` to create imbalance.

Implement a custom `BalancedBatchSampler` that places an equal number of examples from all 8 classes in every batch. `batch_size` must be divisible by 8. Sampling with replacement is allowed for smaller classes. When passing `batch_sampler` to `DataLoader`, do not also pass `batch_size`, `shuffle`, or a separate `sampler`.

Compare it with standard `shuffle=True` batches on the same simulated-imbalanced training set. Keep validation and test unchanged, and compare accuracy, macro-F1, and recall for every class.

## Cross-Entropy vs. Binary Cross-Entropy

For the primary controlled comparison, keep architecture, seed, split, augmentation, optimizer, learning rate, and training budget fixed. If you later compare the best-tuned version of each loss, give both losses equal tuning budgets.

- `CrossEntropyLoss` is the standard choice: the target is an integer class index and the objective models one mutually exclusive class decision.
- `BCEWithLogitsLoss` is an educational one-vs-all experiment: the target must be one-hot encoded and `float`, representing eight independent yes/no decisions.

For BCE, use `argmax(logits)` to get one class for multiclass metrics. Do not apply sigmoid separately before `BCEWithLogitsLoss`. Explain why BCE scores are not necessarily mutually exclusive and do not necessarily sum to one. Compare loss curves, per-class metrics, confidence behaviour, and error patterns.

## Regularization and Learning-Rate Scheduling

Study dropout, weight decay, augmentation, and a scheduler through controlled experiments. Use `AdamW` for the weight-decay experiment when possible. Compare a fixed learning rate with `StepLR` or `ReduceLROnPlateau`.

For each run, report training/validation loss, the train-validation gap, best epoch, macro-F1, and learning rate at every epoch. With `ReduceLROnPlateau`, call `scheduler.step(validation_loss)` after validation. Test performance must not drive any training decision.

## Transfer Learning with ResNet18

Compare these required strategies:

1. **Feature extraction:** freeze an ImageNet-pretrained ResNet18 backbone and train a new 8-class head.
2. **Fine-tuning:** train the head first, then unfreeze `layer4` and the head with a smaller learning rate for pretrained parameters.

For these pretrained comparisons, keep the standard ResNet18 architecture and use ImageNet-compatible resize and normalization. Do not replace `conv1` or remove `maxpool`, because the pretrained first-layer weights would no longer match.

Stretch: train `ResNet18(weights=None)` for small images. In that separate experiment, changing `conv1` and removing `maxpool` is acceptable. Report trainable parameters and learning-rate groups for all transfer-learning runs.

## Evaluation, Precision, Recall, and Error Analysis

Use validation data for experiment comparison. After choosing the final configuration, evaluate it once on the frozen test set. Report overall accuracy, macro precision, macro recall, macro-F1, per-class precision/recall/F1, count-based and row-normalized confusion matrices, training curves, and at least 12 misclassified images with true label, prediction, and confidence.

Create this validation comparison table for the main experiments, then analyze the lowest-precision and lowest-recall class in each meaningful case:

| Experiment | Macro Precision | Macro Recall | Macro F1 | Lowest-Recall Class | Lowest-Precision Class |
|---|---:|---:|---:|---|---|
| CNN baseline |  |  |  |  |  |
| Balanced batches |  |  |  |  |  |
| CrossEntropy / BCE comparison |  |  |  |  |  |
| Best regularized + scheduled model |  |  |  |  |  |
| ResNet feature extraction |  |  |  |  |  |
| ResNet fine-tuning |  |  |  |  |  |

## Class Relationships, `unclean`, and Confidence

Use the row-normalized confusion matrix to rank mutually confused class pairs. A useful starting score is:

```text
pair_confusion(i, j) = C_normalized[i, j] + C_normalized[j, i]
```

Inspect the actual errors, class support, and the practical meaning of labels before proposing a merge. If a pair has substantial mutual confusion and a domain justification, define a merged label mapping, retrain, and compare the original and merged taxonomies. If not, defend the decision not to merge. Retraining a justified merged taxonomy is stretch work.

Do not train on `unclean` initially. First exclude duplicates of train/test images, then inspect label conflicts. Analyze `neysan` as an unseen vehicle class or a human-review case, not as an unexplained ninth training class. Propose `needs_review=True` below a confidence threshold such as `0.70`, but choose and justify the threshold with validation analysis, never with test data.

## Prediction Output and Reproducibility

Implement `predict.py` or an equivalent callable function that accepts an image path and returns JSON:

```json
{
  "predicted_class": "taxi",
  "confidence": 0.87,
  "probabilities": {"ambulance": 0.01, "autobus": 0.02, "kamyun": 0.03, "kamyunet": 0.02, "minibus": 0.01, "savari": 0.02, "taxi": 0.87, "vanet": 0.02},
  "needs_review": false
}
```

The production output should use the selected Cross-Entropy model, whose softmax probabilities sum to approximately one. Keep BCE sigmoid scores in the comparison report rather than presenting them as a final probability distribution. Save checkpoint metadata: architecture, class mapping, transforms, selected threshold, seed, and experiment configuration.

## Deliverables and Grading

Submit a Git repository with a clear README, requirements, reproducible configuration, data-audit report, baseline CNN, controlled ablations, sampler implementation, CE/BCE report, scheduler report, pretrained ResNet comparison, confusion matrices, per-class precision/recall/F1 analysis, `unclean` analysis, and JSON prediction script. Keep datasets and large checkpoints out of Git.

| Component | Points |
|---|---:|
| Repository structure, Git, and README | 5 |
| Data audit and leakage prevention | 10 |
| CNN baseline and training loop | 10 |
| Balanced batches and imbalance analysis | 10 |
| Cross-Entropy vs. BCE comparison | 10 |
| Regularization and learning-rate scheduler | 10 |
| ResNet transfer learning | 20 |
| Evaluation, per-class analysis, confusion matrices, and merge decision | 10 |
| `unclean` and low-confidence analysis | 5 |
| Prediction script, JSON, and reproducibility | 10 |
| **Total** | **100** |

Stretch work: ResNet18 from scratch for small images, retraining a justified merged taxonomy, calibration analysis, mixed precision, Grad-CAM, a FastAPI endpoint, or unit tests.

## Final Checklist

- [ ] Is the class mapping identical between train and test?
- [ ] Was the audit completed before creating validation data and freezing the cleaned test set?
- [ ] Was test used only once for final evaluation?
- [ ] Were duplicates across splits checked with image hashes?
- [ ] Is augmentation applied only to training data?
- [ ] Was the balanced-batch experiment run on a reproducible simulated-imbalanced training set?
- [ ] Does BCE use one-hot `float` targets and `argmax(logits)` for multiclass metrics?
- [ ] Does the scheduler use validation data only, and are learning rates logged?
- [ ] Are accuracy, precision, recall, F1, and both confusion-matrix forms reported?
- [ ] Are per-class precision, recall, and F1 compared across the main experiments?
- [ ] Are the lowest-precision and lowest-recall classes analyzed?
- [ ] Were misclassified images inspected and explained?
- [ ] Was `unclean` analyzed separately after duplicate removal?
- [ ] Does the checkpoint contain class mapping, transforms, threshold, seed, and configuration?
- [ ] Can another person reproduce the final result from the README?